###load_struct_to_prep_reviews
####Builds the Gold layer reviews table (ecom_prep.reviews) — calls Azure AI Language API for multilingual sentiment analysis on Portuguese review text; stores positive, neutral, and negative confidence scores with idempotent processing guard

In [0]:
from azure.ai.textanalytics import TextAnalyticsClient, TextDocumentInput
from azure.core.credentials import AzureKeyCredential
from pyspark.sql import functions as F
import pandas as pd

In [0]:
AI_ENDPOINT = dbutils.secrets.get(scope="e-commerce-secret-scope", key="language-endpoint")
AI_KEY      = dbutils.secrets.get(scope="e-commerce-secret-scope", key="language-key")

client = TextAnalyticsClient(
    endpoint=AI_ENDPOINT,
    credential=AzureKeyCredential(AI_KEY)
)

In [0]:
#Only actual comments will be processed, filtered null comments and empty strings
reviews_df = (
    spark.table("ecom_struct.order_reviews")
    .select("review_id", "order_id", "review_score", "review_comment_message")
    .filter(
        F.col("review_comment_message").isNotNull() &
        (F.trim(F.col("review_comment_message")) != "No Comment") &
        (F.trim(F.col("review_comment_message")) != "")
    )
)

print(f"Reviews to process: {reviews_df.count()}")

#Converting to pandas for sentiment analysis
reviews_pd = reviews_df.toPandas()

In [0]:
#BATCH SENTIMENT ANALYSIS
#Azure AI Language API accepts a maximum of 10 documents per request
#so we process reviews in batches of 10 to stay within API limits
BATCH_SIZE = 10

def analyze_batch(batch: list[tuple]) -> list[dict]:
    """
    Sends a batch of reviews to Azure AI Language for sentiment analysis.
    
    Args:
        batch: list of (review_id, review_text) tuples
    Returns:
        list of dicts with review_id, sentiment label, and confidence scores
    """
    #Build Azure SDK document objects — each needs a unique id, text, and language
    #Portuguese (pt) is passed directly — no translation needed, API handles it natively
    documents = [
        TextDocumentInput(id=str(i), text=row[1], language="pt")
        for i, row in enumerate(batch)
    ]
    
    try:
        #Call Azure AI Language sentiment endpoint
        response = client.analyze_sentiment(documents=documents)
        
        results = []
        for i, doc in enumerate(response):
            review_id = batch[i][0]  #Map back to original review_id using batch index
            
            if not doc.is_error:
                #Successful response — extract sentiment label and confidence scores
                #sentiment label: positive / neutral / negative / mixed
                #confidence scores: three floats that always sum to 1.0
                results.append({
                    "review_id":        review_id,
                    "sentiment":        doc.sentiment,
                    "positive_score":   round(doc.confidence_scores.positive, 4),
                    "neutral_score":    round(doc.confidence_scores.neutral, 4),
                    "negative_score":   round(doc.confidence_scores.negative, 4)
                })
            else:
                # API returned an error for this specific document (e.g. empty text, unsupported chars)
                # Store as 'error' with null scores so the row is still trackable in Gold table
                results.append({
                    "review_id":      review_id,
                    "sentiment":      "error",
                    "positive_score": None,
                    "neutral_score":  None,
                    "negative_score": None
                })
        return results

    except Exception as e:
        # Entire batch failed (e.g. network timeout, auth error)
        # Fail gracefully — return error rows for all reviews in this batch
        # so processing continues for remaining batches
        print(f"Batch failed: {e}")
        return [{
            "review_id":      r[0],
            "sentiment":      "error",
            "positive_score": None,
            "neutral_score":  None,
            "negative_score": None
        } for r in batch]


#RUN BATCHES 
#Zip review_id and comment text into list of tuples for batch processing
rows = list(zip(reviews_pd["review_id"], reviews_pd["review_comment_message"]))
all_results = []

#Iterate through rows in chunks of BATCH_SIZE (10)
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    all_results.extend(analyze_batch(batch))

#SUMMARY
#Count successes vs errors to validate API processing quality
success = sum(1 for r in all_results if r["sentiment"] != "error")
errors  = sum(1 for r in all_results if r["sentiment"] == "error")

print(f"Successfully processed : {success}")
print(f"Errors                 : {errors}")
print(f"Total                  : {len(all_results)}")

In [0]:
#Build Final DataFrame
results_df = spark.createDataFrame(pd.DataFrame(all_results))

reviews = (
    reviews_df
    .join(results_df, on="review_id", how="left")
    .withColumn("processed_at", F.current_timestamp())
)
reviews.createOrReplaceTempView("reviews")

In [0]:
reviews = spark.sql(f'''

select 
review_id,
order_id,
customer_key,
review_score,
review_comment_message,
sentiment,
positive_score,
neutral_score,
negative_score,
review_creation_date,
review_answer_timestamp,
review_processed_date
 from (
select 
r.review_id,
r.order_id,
o.customer_key,
r.review_score,
r.review_comment_message,
r.sentiment,
r.positive_score,
r.neutral_score,
r.negative_score,
date(ord.review_creation_date),
ord.review_answer_timestamp,
r.processed_at as review_processed_date,

row_number() over(partition by r.review_id order by r.processed_at desc) as rn
 from reviews r
left join ecom_prep.orders o 
on r.order_id = o.order_id

left join ecom_struct.order_reviews ord 
on ord.review_id = r.review_id

) where rn = 1 
''')

reviews.createOrReplaceTempView("reviews")

In [0]:
spark.sql("""
    MERGE INTO ecom_prep.reviews AS tgt
    USING (select * from reviews where customer_key is not null ) AS src
    ON tgt.review_id = src.review_id
    WHEN MATCHED THEN UPDATE SET
        tgt.customer_key = src.customer_key,
        tgt.review_score = src.review_score,
        tgt.review_comment_message = src.review_comment_message,
        tgt.sentiment = src.sentiment,
        tgt.positive_score = src.positive_score,
        tgt.neutral_score = src.neutral_score,
        tgt.negative_score = src.negative_score,
        tgt.review_answer_timestamp = src.review_answer_timestamp,
        tgt.review_processed_timestamp = src.review_processed_date
    WHEN NOT MATCHED THEN INSERT (
        review_id,
        order_id,
        customer_key,
        review_score,
        review_comment_message,
        sentiment,
        positive_score,
        neutral_score,
        negative_score,
        review_creation_date,
        review_answer_timestamp,
        review_processed_timestamp
    ) VALUES (
        src.review_id,
        src.order_id,
        src.customer_key,
        src.review_score,
        src.review_comment_message,
        src.sentiment,
        src.positive_score,
        src.neutral_score,
        src.negative_score,
        src.review_creation_date,
        src.review_answer_timestamp,
        src.review_processed_date
    )
""").display()